In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/LLM_FineTuning_Project"

folders = [
    "data/raw",
    "data/processed",
    "data/train",
    "data/validation",
    "data/test",
    "models",
    "results",
    "notebook"
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_DIR, folder), exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("✅ Project structure created")

Project directory: /content/drive/MyDrive/LLM_FineTuning_Project
✅ Project structure created


In [ ]:
import os

print(os.path.exists(PROJECT_DIR))
print(os.listdir(PROJECT_DIR))

True
['data', 'models', 'results', 'notebook']


In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "DuoNeural/ml-ai-engineer-sft"
)

print(dataset)

README.md:   0%|          | 0.00/7.05k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/1.75M [00:00<?, ?B/s]

validation.jsonl:   0%|          | 0.00/90.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1608 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/84 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response', 'topic', 'difficulty'],
        num_rows: 1608
    })
    validation: Dataset({
        features: ['instruction', 'response', 'topic', 'difficulty'],
        num_rows: 84
    })
})


In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response', 'topic', 'difficulty'],
        num_rows: 1608
    })
    validation: Dataset({
        features: ['instruction', 'response', 'topic', 'difficulty'],
        num_rows: 84
    })
})


In [ ]:
print(dataset[list(dataset.keys())[0]][0])

{'instruction': "For a new vision transformer pre-training run on 100M images, we're considering using a `0.02 * torch.randn(...)` for all attention and feed-forward linear layers, and `torch.zeros(...)` for biases. Standard AdamW, 16-bit bfloat16. What's your take?", 'response': "`0.02 * torch.randn` is a common heuristic, but it's not robustly principled for deep ViTs. It can lead to vanishing gradients in deeper layers if the activations are too small, or slow convergence. For ViTs, you should be using a truncated normal distribution, often `nn.init.trunc_normal_` (or `torch.nn.init.trunc_normal_` if available in your PyTorch version, otherwise adapt from `timm` or `transformers` libraries) with a specific `std` determined by the layer configuration (e.g., `std=0.02` can be a starting point, but consider scaling it with `1/sqrt(fan_in)` for certain sub-layers). Truncation is critical to prevent outlier weights that can destabilize BF16 training. Also, check the specific ViT architec

In [ ]:
split_name = list(dataset.keys())[0]

print("Split:", split_name)
print("Number of examples:", len(dataset[split_name]))
print("Columns:", dataset[split_name].column_names)

Split: train
Number of examples: 1608
Columns: ['instruction', 'response', 'topic', 'difficulty']


In [ ]:
example = dataset[split_name][0]

for key, value in example.items():
    print(f"\n--- {key} ---")
    print(value)


--- instruction ---
For a new vision transformer pre-training run on 100M images, we're considering using a `0.02 * torch.randn(...)` for all attention and feed-forward linear layers, and `torch.zeros(...)` for biases. Standard AdamW, 16-bit bfloat16. What's your take?

--- response ---
`0.02 * torch.randn` is a common heuristic, but it's not robustly principled for deep ViTs. It can lead to vanishing gradients in deeper layers if the activations are too small, or slow convergence. For ViTs, you should be using a truncated normal distribution, often `nn.init.trunc_normal_` (or `torch.nn.init.trunc_normal_` if available in your PyTorch version, otherwise adapt from `timm` or `transformers` libraries) with a specific `std` determined by the layer configuration (e.g., `std=0.02` can be a starting point, but consider scaling it with `1/sqrt(fan_in)` for certain sub-layers). Truncation is critical to prevent outlier weights that can destabilize BF16 training. Also, check the specific ViT a

In [ ]:
from collections import Counter

train_data = dataset["train"]

print("Total examples:", len(train_data))

print("\nTopics:")
topic_counts = Counter(train_data["topic"])

for topic, count in topic_counts.most_common():
    print(f"{count:4d} | {topic}")

print("\nDifficulty:")
difficulty_counts = Counter(train_data["difficulty"])

for difficulty, count in difficulty_counts.most_common():
    print(f"{count:4d} | {difficulty}")

Total examples: 1608

Topics:
  36 | Mixed precision training pitfalls (fp16 vs bf16, loss scaling, gradient underflow)
  36 | Checkpoint resume bugs (optimizer state mismatch, torch.compile state_dict prefix issues, LR schedule discontinuity)
  36 | Critiquing a novel architecture proposal for missing ablations or confounds
  36 | Calibration dataset choice for post-training quantization
  36 | GRPO vs PPO tradeoffs for LLM fine-tuning
  35 | Weight initialization schemes and why they matter at scale
  35 | Gradient clipping — when, why, and what threshold
  35 | Overfitting vs underfitting diagnosis from loss curves alone
  35 | Curriculum learning and data ordering effects
  35 | Designing a red-team eval set for a new model release
  35 | Cost/latency tradeoffs in model selection for a product feature
  35 | Custom CUDA kernel debugging basics and when you actually need one
  35 | Tradeoffs between dense vs sparse (MoE) models at a given inference budget
  35 | State space models /

In [ ]:
print("\nExample lengths:")

instruction_lengths = [
    len(x["instruction"].split())
    for x in train_data
]

response_lengths = [
    len(x["response"].split())
    for x in train_data
]

print("Average instruction words:", sum(instruction_lengths) / len(instruction_lengths))
print("Average response words:", sum(response_lengths) / len(response_lengths))
print("Longest instruction:", max(instruction_lengths))
print("Longest response:", max(response_lengths))


Example lengths:
Average instruction words: 37.6044776119403
Average response words: 101.56218905472637
Longest instruction: 155
Longest response: 487


In [ ]:
instructions = train_data["instruction"]

unique_instructions = set(instructions)

print("Total instructions:", len(instructions))
print("Unique instructions:", len(unique_instructions))
print("Duplicate instructions:", len(instructions) - len(unique_instructions))

Total instructions: 1608
Unique instructions: 1608
Duplicate instructions: 0


In [ ]:
responses = train_data["response"]

unique_responses = set(responses)

print("\nTotal responses:", len(responses))
print("Unique responses:", len(unique_responses))
print("Duplicate responses:", len(responses) - len(unique_responses))


Total responses: 1608
Unique responses: 1608
Duplicate responses: 0


In [ ]:
for column in train_data.column_names:
    missing = sum(
        1 for value in train_data[column]
        if value is None or str(value).strip() == ""
    )

    print(f"{column}: {missing} missing")

instruction: 0 missing
response: 0 missing
topic: 0 missing
difficulty: 0 missing


In [ ]:
from sklearn.model_selection import train_test_split

# Convert Hugging Face dataset to pandas
df = train_data.to_pandas()

print("Original dataset:", len(df))
print(df.head())

Original dataset: 1608
                                         instruction  \
0  For a new vision transformer pre-training run ...   
1  Training Llama-2-7B with FP16, dynamic loss sc...   
2  I'm trying to load an int8 PTQ model saved wit...   
3  My custom MoE implementation logs `expert_load...   
4  I have a custom `LlamaRMSNorm` implementation....   

                                            response  \
0  `0.02 * torch.randn` is a common heuristic, bu...   
1  You're applying gradient clipping *before* `sc...   
2  You're saving the state dict of a quantized mo...   
3  This discrepancy indicates that while the *tot...   
4  The problem is `hidden_states = hidden_states....   

                                               topic difficulty  
0  Weight initialization schemes and why they mat...      staff  
1  Mixed precision training pitfalls (fp16 vs bf1...      staff  
2  Quantization-aware training vs post-training q...      staff  
3  Mixture-of-Experts routing design ch

In [ ]:
# Create a combined stratification label
df["stratify_label"] = (
    df["topic"].astype(str) + "___" +
    df["difficulty"].astype(str)
)

print("Unique stratification groups:", df["stratify_label"].nunique())

Unique stratification groups: 141


In [ ]:
# 80% train, 20% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["stratify_label"]
)

# Split remaining 20% into 10% validation and 10% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["stratify_label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 1286
Validation: 161
Test: 161


In [ ]:
for df_split in [train_df, val_df, test_df]:
    df_split.drop(columns=["stratify_label"], inplace=True)

print(train_df.columns.tolist())

['instruction', 'response', 'topic', 'difficulty']


In [ ]:
import os

train_path = os.path.join(PROJECT_DIR, "data/train/train.csv")
val_path = os.path.join(PROJECT_DIR, "data/validation/validation.csv")
test_path = os.path.join(PROJECT_DIR, "data/test/test.csv")

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print("✅ Dataset splits saved!")
print(train_path)
print(val_path)
print(test_path)

✅ Dataset splits saved!
/content/drive/MyDrive/LLM_FineTuning_Project/data/train/train.csv
/content/drive/MyDrive/LLM_FineTuning_Project/data/validation/validation.csv
/content/drive/MyDrive/LLM_FineTuning_Project/data/test/test.csv


In [ ]:
print("Train exists:", os.path.exists(train_path))
print("Validation exists:", os.path.exists(val_path))
print("Test exists:", os.path.exists(test_path))

Train exists: True
Validation exists: True
Test exists: True


In [ ]:
import pandas as pd

train_check = pd.read_csv(train_path)
val_check = pd.read_csv(val_path)
test_check = pd.read_csv(test_path)

print("TRAIN:", train_check.shape)
print("VALIDATION:", val_check.shape)
print("TEST:", test_check.shape)

print("\nColumns:")
print(train_check.columns.tolist())

TRAIN: (1286, 4)
VALIDATION: (161, 4)
TEST: (161, 4)

Columns:
['instruction', 'response', 'topic', 'difficulty']


In [ ]:
print("\nSample from TRAIN:")
print(train_check.iloc[0])

print("\nSample from VALIDATION:")
print(val_check.iloc[0])

print("\nSample from TEST:")
print(test_check.iloc[0])


Sample from TRAIN:
instruction    I'm setting up a Prometheus gauge for our mode...
response       The `NaN`s are coming from `0 * log(0)` or `lo...
topic          Common interview question: design a system to ...
difficulty                                                 staff
Name: 0, dtype: object

Sample from VALIDATION:
instruction    My colleague wrote this for loading data. Any ...
response       This is terrible for reproducibility and porta...
topic          Versioning and reproducibility for ML experiments
difficulty                                                 staff
Name: 0, dtype: object

Sample from TEST:
instruction    Is there *any* scenario where a learned absolu...
response       Yes, for specific, fixed-length sequence model...
topic          Positional encoding schemes (RoPE, ALiBi, lear...
difficulty                                                senior
Name: 0, dtype: object


In [ ]:
def format_example(row):
    return (
        "### Instruction:\n"
        + row["instruction"].strip()
        + "\n\n"
        "### Response:\n"
        + row["response"].strip()
    )

train_check["text"] = train_check.apply(format_example, axis=1)
val_check["text"] = val_check.apply(format_example, axis=1)
test_check["text"] = test_check.apply(format_example, axis=1)

print(train_check["text"].iloc[0])

### Instruction:
I'm setting up a Prometheus gauge for our model's prediction entropy. I'm seeing `NaN` values sometimes when the model is very confident and outputs probabilities close to 0 or 1. How should I handle this in the metric calculation?

### Response:
The `NaN`s are coming from `0 * log(0)` or `log(0)` if your probabilities are exactly 0. Standard entropy formulas define `0 * log(0)` as `0`. You need to handle this explicitly. When calculating `p_i * log(p_i)`: if `p_i` is exactly 0, set the term to `0`. If your probabilities can legitimately be exactly 1, `1 * log(1)` is `0`, which is correctly handled by most log implementations. Do *not* add a generic small epsilon to *all* probabilities as a blanket fix, as this will artificially increase entropy. Only handle the `p_i=0` case by replacing the term with `0`.


In [ ]:
print("Train examples:", len(train_check))
print("Validation examples:", len(val_check))
print("Test examples:", len(test_check))

print("\nColumns:")
print(train_check.columns.tolist())

Train examples: 1286
Validation examples: 161
Test examples: 161

Columns:
['instruction', 'response', 'topic', 'difficulty', 'text']


In [ ]:
for split_name, split_df in [
    ("TRAIN", train_check),
    ("VALIDATION", val_check),
    ("TEST", test_check)
]:
    empty = split_df["text"].isna().sum()
    print(f"{split_name} empty text:", empty)

TRAIN empty text: 0
VALIDATION empty text: 0
TEST empty text: 0


In [ ]:
formatted_train_path = os.path.join(
    PROJECT_DIR, "data/processed/train_formatted.csv"
)

formatted_val_path = os.path.join(
    PROJECT_DIR, "data/processed/validation_formatted.csv"
)

formatted_test_path = os.path.join(
    PROJECT_DIR, "data/processed/test_formatted.csv"
)

train_check.to_csv(formatted_train_path, index=False)
val_check.to_csv(formatted_val_path, index=False)
test_check.to_csv(formatted_test_path, index=False)

print("✅ Formatted datasets saved!")
print(formatted_train_path)
print(formatted_val_path)
print(formatted_test_path)

✅ Formatted datasets saved!
/content/drive/MyDrive/LLM_FineTuning_Project/data/processed/train_formatted.csv
/content/drive/MyDrive/LLM_FineTuning_Project/data/processed/validation_formatted.csv
/content/drive/MyDrive/LLM_FineTuning_Project/data/processed/test_formatted.csv


In [ ]:
print(
    "Train:",
    os.path.exists(formatted_train_path)
)

print(
    "Validation:",
    os.path.exists(formatted_val_path)
)

print(
    "Test:",
    os.path.exists(formatted_test_path)
)

Train: True
Validation: True
Test: True


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("⚠️ GPU not detected")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:
import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/LLM_FineTuning_Project"

print("Project exists:", os.path.exists(PROJECT_DIR))
print("Project contents:", os.listdir(PROJECT_DIR))

Project exists: True
Project contents: ['data', 'models', 'results', 'notebook']


In [ ]:
train_path = f"{PROJECT_DIR}/data/processed/train_formatted.csv"
val_path = f"{PROJECT_DIR}/data/processed/validation_formatted.csv"
test_path = f"{PROJECT_DIR}/data/processed/test_formatted.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (1286, 5)
Validation: (161, 5)
Test: (161, 5)


In [ ]:
print(train_df.columns.tolist())
print("\nExample:")
print(train_df["text"].iloc[0])

['instruction', 'response', 'topic', 'difficulty', 'text']

Example:
### Instruction:
I'm setting up a Prometheus gauge for our model's prediction entropy. I'm seeing `NaN` values sometimes when the model is very confident and outputs probabilities close to 0 or 1. How should I handle this in the metric calculation?

### Response:
The `NaN`s are coming from `0 * log(0)` or `log(0)` if your probabilities are exactly 0. Standard entropy formulas define `0 * log(0)` as `0`. You need to handle this explicitly. When calculating `p_i * log(p_i)`: if `p_i` is exactly 0, set the term to `0`. If your probabilities can legitimately be exactly 1, `1 * log(1)` is `0`, which is correctly handled by most log implementations. Do *not* add a generic small epsilon to *all* probabilities as a blanket fix, as this will artificially increase entropy. Only handle the `p_i=0` case by replacing the term with `0`.


In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 22.4 MB/s eta 0:00:00


In [ ]:
!pip install -q --upgrade --force-reinstall "pyarrow==21.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 23.7 MB/s eta 0:00:00


In [ ]:
import pyarrow
import datasets

print("PyArrow:", pyarrow.__version__)
print("Datasets:", datasets.__version__)

PyArrow: 21.0.0
Datasets: 5.0.1


In [ ]:
import transformers
import peft
import accelerate
import bitsandbytes
import trl

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("TRL:", trl.__version__)

Transformers: 5.15.0
PEFT: 0.20.0
Accelerate: 1.14.0
BitsAndBytes: 0.50.0
TRL: 1.9.2


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("✅ Model loaded")
print("Model:", MODEL_NAME)
print("Device:", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Model loaded
Model: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0


In [ ]:
import torch

question = test_df["instruction"].iloc[0]

messages = [
    {
        "role": "user",
        "content": question
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("QUESTION:")
print(question)

print("\nBASE MODEL ANSWER:")
print(answer)

print("\nREFERENCE ANSWER:")
print(test_df["response"].iloc[0])

QUESTION:
Is there *any* scenario where a learned absolute positional embedding is a better choice than RoPE or ALiBi, even knowing its extrapolation limitations?

BASE MODEL ANSWER:
Positional embeddings are crucial in many natural language processing (NLP) tasks because they allow the model to understand the relative positions of words within sentences. There are several types of positional encodings that have been proposed and used in various NLP models:

1. **Learned Absolute Positional Embeddings**: These are learned parameters that represent each position in the sequence independently.
2. **Rotary Positional Encoding (RoPE)**: This approach uses sinusoidal functions to encode the relative positions between tokens.
3. **ALiBi (Attention Layer Biases)**: Similar to RoPE but with different biases for attention.

### When Learned Absolute Positional Embeddings Might Be Better

While RoPE and ALiBi are more sophisticated and often outperform learned absolute positional embeddings on c

In [ ]:
!pip install -q evaluate rouge_score bert_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.0 MB/s eta 0:00:00


In [ ]:
import evaluate

rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

print("✅ Evaluation metrics loaded")

✅ Evaluation metrics loaded


In [ ]:
import torch
import pandas as pd
from tqdm.auto import tqdm

model.eval()

baseline_records = []

for i, row in tqdm(test_df.iterrows(), total=len(test_df)):

    messages = [
        {
            "role": "user",
            "content": row["instruction"]
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    baseline_records.append({
        "index": i,
        "instruction": row["instruction"],
        "reference": row["response"],
        "prediction": prediction,
        "topic": row["topic"],
        "difficulty": row["difficulty"]
    })

print("✅ Baseline generation complete!")
print("Examples generated:", len(baseline_records))

  0%|          | 0/161 [00:00<?, ?it/s]

✅ Baseline generation complete!
Examples generated: 161


#Calculate metrics for the base qwen

In [ ]:
predictions = [x["prediction"] for x in baseline_records]
references = [x["reference"] for x in baseline_records]

rouge_result = rouge.compute(
    predictions=predictions,
    references=references
)

print("ROUGE results:")
print(rouge_result)

ROUGE results:
{'rouge1': np.float64(0.23881541833481645), 'rouge2': np.float64(0.04357248563060857), 'rougeL': np.float64(0.11450715947124623), 'rougeLsum': np.float64(0.1714869595829438)}


In [ ]:
bertscore_result = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="en",
    device="cuda"
)

baseline_bertscore = sum(bertscore_result["f1"]) / len(
    bertscore_result["f1"]
)

print("Average BERTScore F1:", baseline_bertscore)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Average BERTScore F1: 0.8187245408200329


In [ ]:
results_dir = os.path.join(PROJECT_DIR, "results")
os.makedirs(results_dir, exist_ok=True)

baseline_df = pd.DataFrame(baseline_records)

baseline_predictions_path = os.path.join(
    results_dir,
    "baseline_predictions.csv"
)

baseline_df.to_csv(
    baseline_predictions_path,
    index=False
)

baseline_metrics = {
    "model": MODEL_NAME,
    "num_test_examples": len(baseline_df),
    "rouge1": rouge_result["rouge1"],
    "rouge2": rouge_result["rouge2"],
    "rougeL": rouge_result["rougeL"],
    "bertscore_f1": baseline_bertscore
}

baseline_metrics_path = os.path.join(
    results_dir,
    "baseline_metrics.csv"
)

pd.DataFrame([baseline_metrics]).to_csv(
    baseline_metrics_path,
    index=False
)

print("✅ Baseline predictions saved:")
print(baseline_predictions_path)

print("\n✅ Baseline metrics saved:")
print(baseline_metrics_path)

print("\nBASELINE RESULTS")
for key, value in baseline_metrics.items():
    print(f"{key}: {value}")

✅ Baseline predictions saved:
/content/drive/MyDrive/LLM_FineTuning_Project/results/baseline_predictions.csv

✅ Baseline metrics saved:
/content/drive/MyDrive/LLM_FineTuning_Project/results/baseline_metrics.csv

BASELINE RESULTS
model: Qwen/Qwen2.5-1.5B-Instruct
num_test_examples: 161
rouge1: 0.23881541833481645
rouge2: 0.04357248563060857
rougeL: 0.11450715947124623
bertscore_f1: 0.8187245408200329


#Dataset ready for fine tuning

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "GPU memory reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

GPU memory allocated: 3.88 GB
GPU memory reserved: 3.91 GB


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["text"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["text"]],
    preserve_index=False
)

print("Train:", train_dataset)
print("Validation:", val_dataset)

Train: Dataset({
    features: ['text'],
    num_rows: 1286
})
Validation: Dataset({
    features: ['text'],
    num_rows: 161
})


In [ ]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("✅ 4-bit QLoRA configuration created")

✅ 4-bit QLoRA configuration created


#Reload Qwen using 4-bit

In [ ]:
import gc
import torch

del model
gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

GPU allocated: 1.0 GB


In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ 4-bit model loaded")
print("Device:", model.device)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ 4-bit model loaded
Device: cuda:0


In [ ]:
print(
    "GPU allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "GPU reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

GPU allocated: 2.08 GB
GPU reserved: 3.88 GB


#k-bit training and attach the LoRA adapters.

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

print("✅ Model prepared for k-bit training")

✅ Model prepared for k-bit training


In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

print("✅ LoRA configuration created")

✅ LoRA configuration created


In [ ]:
from peft import get_peft_model

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=f"{PROJECT_DIR}/models/qwen2.5-1.5b-qlora",

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,

    # Validation
    per_device_eval_batch_size=2,
    eval_strategy="steps",
    eval_steps=50,

    # Optimization
    learning_rate=2e-4,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,

    # Precision / memory
    bf16=True,
    gradient_checkpointing=True,

    # Logging
    logging_strategy="steps",
    logging_steps=10,
    report_to="none",

    # Checkpoints
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    # Select best checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Reproducibility
    seed=42
)

print("✅ Training configuration created")

✅ Training configuration created


In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id

print("PAD token:", tokenizer.pad_token)
print("PAD token ID:", tokenizer.pad_token_id)

PAD token: <|endoftext|>
PAD token ID: 151643


In [ ]:
print("Tokenizer vocab size:", len(tokenizer))
print("Model max position embeddings:",
      getattr(model.config, "max_position_embeddings", "N/A"))

Tokenizer vocab size: 151665
Model max position embeddings: 32768


In [ ]:
def get_token_length(text):
    return len(tokenizer(text, add_special_tokens=True)["input_ids"])

train_token_lengths = [
    get_token_length(text)
    for text in train_df["text"]
]

val_token_lengths = [
    get_token_length(text)
    for text in val_df["text"]
]

test_token_lengths = [
    get_token_length(text)
    for text in test_df["text"]
]

print("TRAIN")
print("Average:", sum(train_token_lengths) / len(train_token_lengths))
print("Maximum:", max(train_token_lengths))

print("\nVALIDATION")
print("Average:", sum(val_token_lengths) / len(val_token_lengths))
print("Maximum:", max(val_token_lengths))

print("\nTEST")
print("Average:", sum(test_token_lengths) / len(test_token_lengths))
print("Maximum:", max(test_token_lengths))

TRAIN
Average: 218.16407465007777
Maximum: 1109

VALIDATION
Average: 223.06832298136646
Maximum: 444

TEST
Average: 220.7639751552795
Maximum: 466


In [ ]:
import numpy as np

for percentile in [90, 95, 99]:
    print(
        f"{percentile}th percentile:",
        int(np.percentile(train_token_lengths, percentile))
    )

90th percentile: 294
95th percentile: 334
99th percentile: 419


In [ ]:
MAX_LENGTH = 512

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("✅ Tokenization complete")
print("Train:", tokenized_train)
print("Validation:", tokenized_val)

Map:   0%|          | 0/1286 [00:00<?, ? examples/s]

Map:   0%|          | 0/161 [00:00<?, ? examples/s]

✅ Tokenization complete
Train: Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 1286
})
Validation: Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 161
})


In [ ]:
print("Example token count:",
      len(tokenized_train[0]["input_ids"]))

print("Maximum train token count:",
      max(len(x) for x in tokenized_train["input_ids"]))

print("Maximum validation token count:",
      max(len(x) for x in tokenized_val["input_ids"]))

Example token count: 207
Maximum train token count: 512
Maximum validation token count: 444


#Create the SFT Trainer

#Create the trainer

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer
)

print("✅ SFTTrainer created successfully")

Building labels for train dataset:   0%|          | 0/1286 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1286 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1286 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/161 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/161 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/161 [00:00<?, ? examples/s]

✅ SFTTrainer created successfully


#Final pre-training sanity check

In [ ]:
print("Trainable parameters:")
model.print_trainable_parameters()

print("\nGPU memory:")
print(
    "Allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)
print(
    "Reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

print("\nTraining examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))

Trainable parameters:
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

GPU memory:
Allocated: 2.55 GB
Reserved: 3.96 GB

Training examples: 1286
Validation examples: 161


#Start QLoRA fine-tuning

In [ ]:
print("🚀 Starting QLoRA fine-tuning...")
print("Training examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))

train_result = trainer.train()

print("\n✅ Training completed!")
print("Final training loss:", train_result.training_loss)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🚀 Starting QLoRA fine-tuning...
Training examples: 1286
Validation examples: 161


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.960788,1.942158,1.907582,174592.000000,0.541780
100,1.725992,1.881339,1.709860,344433.000000,0.551143
150,1.694884,1.843237,1.727581,519881.000000,0.556169
200,1.520363,1.866593,1.562161,692515.000000,0.555349
243,1.504986,1.865323,1.561125,839337.000000,0.555560



✅ Training completed!
Final training loss: 1.7608994574213224


In [ ]:
import os

adapter_dir = os.path.join(
    PROJECT_DIR,
    "models",
    "qwen2.5-1.5b-qlora-final"
)

os.makedirs(adapter_dir, exist_ok=True)

trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print("✅ Fine-tuned adapter saved!")
print(adapter_dir)

✅ Fine-tuned adapter saved!
/content/drive/MyDrive/LLM_FineTuning_Project/models/qwen2.5-1.5b-qlora-final


In [ ]:
print("Saved files:")
print(os.listdir(adapter_dir))

Saved files:
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin']


#Generate fine-tuned answers

In [ ]:
import torch
from tqdm.auto import tqdm

model.eval()

finetuned_records = []

for i, row in tqdm(test_df.iterrows(), total=len(test_df)):

    messages = [
        {
            "role": "user",
            "content": row["instruction"]
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    finetuned_records.append({
        "index": i,
        "instruction": row["instruction"],
        "reference": row["response"],
        "prediction": prediction,
        "topic": row["topic"],
        "difficulty": row["difficulty"]
    })

print("✅ Fine-tuned generation complete!")
print("Examples generated:", len(finetuned_records))

  0%|          | 0/161 [00:00<?, ?it/s]

✅ Fine-tuned generation complete!
Examples generated: 161


In [ ]:
finetuned_predictions = [
    x["prediction"] for x in finetuned_records
]

finetuned_references = [
    x["reference"] for x in finetuned_records
]

finetuned_rouge = rouge.compute(
    predictions=finetuned_predictions,
    references=finetuned_references
)

print("Fine-tuned ROUGE:")
print(finetuned_rouge)

Fine-tuned ROUGE:
{'rouge1': np.float64(0.3239271677994099), 'rouge2': np.float64(0.06006963426198367), 'rougeL': np.float64(0.14934253815906987), 'rougeLsum': np.float64(0.16227307320941947)}


In [ ]:
finetuned_bertscore_result = bertscore.compute(
    predictions=finetuned_predictions,
    references=finetuned_references,
    lang="en",
    device="cuda"
)

finetuned_bertscore = sum(
    finetuned_bertscore_result["f1"]
) / len(finetuned_bertscore_result["f1"])

print("Fine-tuned BERTScore F1:", finetuned_bertscore)

Fine-tuned BERTScore F1: 0.8497241365243189


In [ ]:
finetuned_df = pd.DataFrame(finetuned_records)

finetuned_predictions_path = os.path.join(
    results_dir,
    "finetuned_predictions.csv"
)

finetuned_df.to_csv(
    finetuned_predictions_path,
    index=False
)

finetuned_metrics = {
    "model": "Qwen2.5-1.5B-Instruct-QLoRA",
    "num_test_examples": len(finetuned_df),
    "rouge1": finetuned_rouge["rouge1"],
    "rouge2": finetuned_rouge["rouge2"],
    "rougeL": finetuned_rouge["rougeL"],
    "bertscore_f1": finetuned_bertscore
}

finetuned_metrics_path = os.path.join(
    results_dir,
    "finetuned_metrics.csv"
)

pd.DataFrame([finetuned_metrics]).to_csv(
    finetuned_metrics_path,
    index=False
)

print("✅ Fine-tuned predictions saved:")
print(finetuned_predictions_path)

print("\n✅ Fine-tuned metrics saved:")
print(finetuned_metrics_path)

print("\nFINE-TUNED RESULTS")
for key, value in finetuned_metrics.items():
    print(f"{key}: {value}")

✅ Fine-tuned predictions saved:
/content/drive/MyDrive/LLM_FineTuning_Project/results/finetuned_predictions.csv

✅ Fine-tuned metrics saved:
/content/drive/MyDrive/LLM_FineTuning_Project/results/finetuned_metrics.csv

FINE-TUNED RESULTS
model: Qwen2.5-1.5B-Instruct-QLoRA
num_test_examples: 161
rouge1: 0.3239271677994099
rouge2: 0.06006963426198367
rougeL: 0.14934253815906987
bertscore_f1: 0.8497241365243189


#Quantify the improvement

In [ ]:
import pandas as pd
import os

baseline = pd.read_csv(
    os.path.join(results_dir, "baseline_metrics.csv")
)

finetuned = pd.read_csv(
    os.path.join(results_dir, "finetuned_metrics.csv")
)

metrics = ["rouge1", "rouge2", "rougeL", "bertscore_f1"]

comparison = pd.DataFrame({
    "Metric": metrics,
    "Baseline": [baseline.loc[0, m] for m in metrics],
    "Fine-tuned": [finetuned.loc[0, m] for m in metrics]
})

comparison["Absolute Improvement"] = (
    comparison["Fine-tuned"] - comparison["Baseline"]
)

comparison["Relative Improvement (%)"] = (
    comparison["Absolute Improvement"]
    / comparison["Baseline"]
    * 100
)

print(comparison.to_string(index=False))

      Metric  Baseline  Fine-tuned  Absolute Improvement  Relative Improvement (%)
      rouge1  0.238815    0.323927              0.085112                 35.639135
      rouge2  0.043572    0.060070              0.016497                 37.861390
      rougeL  0.114507    0.149343              0.034835                 30.422009
bertscore_f1  0.818725    0.849724              0.031000                  3.786328


In [ ]:
comparison_path = os.path.join(
    results_dir,
    "baseline_vs_finetuned.csv"
)

comparison.to_csv(
    comparison_path,
    index=False
)

print("✅ Comparison saved:")
print(comparison_path)

✅ Comparison saved:
/content/drive/MyDrive/LLM_FineTuning_Project/results/baseline_vs_finetuned.csv


In [ ]:
results = pd.read_csv(
    os.path.join(results_dir, "finetuned_predictions.csv")
)

print("Total examples:", len(results))

for i in range(3):
    row = results.iloc[i]

    print("\n" + "=" * 80)
    print("EXAMPLE", i + 1)
    print("=" * 80)

    print("\nQUESTION:")
    print(row["instruction"])

    print("\nBASE MODEL:")
    print(
        pd.read_csv(
            os.path.join(results_dir, "baseline_predictions.csv")
        ).iloc[i]["prediction"]
    )

    print("\nFINE-TUNED MODEL:")
    print(row["prediction"])

    print("\nREFERENCE:")
    print(row["reference"])

Total examples: 161

EXAMPLE 1

QUESTION:
Is there *any* scenario where a learned absolute positional embedding is a better choice than RoPE or ALiBi, even knowing its extrapolation limitations?

BASE MODEL:
Positional embeddings are crucial in many natural language processing (NLP) tasks because they allow the model to understand the relative positions of words within sentences. There are several types of positional encodings that have been proposed and used in various NLP models:

1. **Learned Absolute Positional Embeddings**: These are learned parameters that represent each position in the sequence independently.
2. **Rotary Positional Encoding (RoPE)**: This approach uses sinusoidal functions to encode the relative positions between tokens.
3. **ALiBi (Attention Layer Biases)**: Similar to RoPE but with different biases for attention.

### When Learned Absolute Positional Embeddings Might Be Better

While RoPE and ALiBi are more sophisticated and often outperform learned absolute p

#Per-example error analysis

In [ ]:
from rouge_score import rouge_scorer
import numpy as np

scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

baseline_scores = []
finetuned_scores = []

for i in range(len(baseline_df)):
    reference = baseline_df.iloc[i]["reference"]

    baseline_prediction = baseline_df.iloc[i]["prediction"]
    finetuned_prediction = finetuned_df.iloc[i]["prediction"]

    baseline_score = scorer.score(
        reference,
        baseline_prediction
    )["rougeL"].fmeasure

    finetuned_score = scorer.score(
        reference,
        finetuned_prediction
    )["rougeL"].fmeasure

    baseline_scores.append(baseline_score)
    finetuned_scores.append(finetuned_score)

error_analysis = pd.DataFrame({
    "index": range(len(baseline_df)),
    "instruction": baseline_df["instruction"],
    "topic": baseline_df["topic"],
    "difficulty": baseline_df["difficulty"],
    "baseline_rougeL": baseline_scores,
    "finetuned_rougeL": finetuned_scores
})

error_analysis["improvement"] = (
    error_analysis["finetuned_rougeL"]
    - error_analysis["baseline_rougeL"]
)

print("✅ Per-example analysis complete")
print("\nAverage baseline ROUGE-L:",
      error_analysis["baseline_rougeL"].mean())

print("Average fine-tuned ROUGE-L:",
      error_analysis["finetuned_rougeL"].mean())

print("Average improvement:",
      error_analysis["improvement"].mean())

✅ Per-example analysis complete

Average baseline ROUGE-L: 0.12315542062104184
Average fine-tuned ROUGE-L: 0.15874765206889332
Average improvement: 0.03559223144785148


In [ ]:
print("\n📈 TOP 5 IMPROVEMENTS")
print(
    error_analysis
    .sort_values("improvement", ascending=False)
    [["index", "topic", "difficulty",
      "baseline_rougeL", "finetuned_rougeL", "improvement"]]
    .head(5)
    .to_string(index=False)
)

print("\n📉 TOP 5 REGRESSIONS")
print(
    error_analysis
    .sort_values("improvement", ascending=True)
    [["index", "topic", "difficulty",
      "baseline_rougeL", "finetuned_rougeL", "improvement"]]
    .head(5)
    .to_string(index=False)
)


📈 TOP 5 IMPROVEMENTS
 index                                                                      topic difficulty  baseline_rougeL  finetuned_rougeL  improvement
     1                            KV cache memory math and context length scaling      staff         0.139630          0.266667     0.127036
    50               Mixture-of-Experts routing design choices and load balancing     senior         0.108000          0.234375     0.126375
   131 Batch size / gradient accumulation tradeoffs and effective batch size math     junior         0.128266          0.240964     0.112698
    71              Explaining backpropagation through time and its failure modes      staff         0.093126          0.202765     0.109639
   142                                   DPO/IPO preference optimization pitfalls      staff         0.101961          0.205980     0.104019

📉 TOP 5 REGRESSIONS
 index                                                                                   topic difficulty  base

In [ ]:
difficulty_analysis = (
    error_analysis
    .groupby("difficulty")
    .agg(
        examples=("index", "count"),
        baseline_rougeL=("baseline_rougeL", "mean"),
        finetuned_rougeL=("finetuned_rougeL", "mean"),
        average_improvement=("improvement", "mean")
    )
    .reset_index()
)

difficulty_analysis["relative_improvement_%"] = (
    difficulty_analysis["average_improvement"]
    / difficulty_analysis["baseline_rougeL"]
    * 100
)

print(
    difficulty_analysis
    .sort_values("average_improvement", ascending=False)
    .to_string(index=False)
)

difficulty  examples  baseline_rougeL  finetuned_rougeL  average_improvement  relative_improvement_%
     staff        52         0.119954          0.161741             0.041786               34.835320
    senior        54         0.121988          0.156644             0.034656               28.409795
    junior        55         0.127329          0.157983             0.030655               24.075255


In [ ]:
topic_analysis = (
    error_analysis
    .groupby("topic")
    .agg(
        examples=("index", "count"),
        baseline_rougeL=("baseline_rougeL", "mean"),
        finetuned_rougeL=("finetuned_rougeL", "mean"),
        average_improvement=("improvement", "mean")
    )
    .reset_index()
)

topic_analysis["relative_improvement_%"] = (
    topic_analysis["average_improvement"]
    / topic_analysis["baseline_rougeL"]
    * 100
)

print(
    topic_analysis
    .sort_values("average_improvement", ascending=False)
    .to_string(index=False)
)

                                                                                                               topic  examples  baseline_rougeL  finetuned_rougeL  average_improvement  relative_improvement_%
                                          Batch size / gradient accumulation tradeoffs and effective batch size math         2         0.129439          0.219861             0.090422               69.856514
                                                                             Reward hacking diagnosis and mitigation         4         0.147336          0.215850             0.068514               46.501540
                                                        Overfitting vs underfitting diagnosis from loss curves alone         3         0.122190          0.189270             0.067080               54.897968
                                        Designing a training job that survives preemption/spot instance interruption         3         0.108335          0.172207           

#Analyze the worst regressions

In [ ]:
worst_indices = (
    error_analysis
    .sort_values("improvement", ascending=True)
    .head(5)["index"]
    .tolist()
)

baseline_full = pd.read_csv(
    os.path.join(results_dir, "baseline_predictions.csv")
)

finetuned_full = pd.read_csv(
    os.path.join(results_dir, "finetuned_predictions.csv")
)

for idx in worst_indices:

    b = baseline_full.iloc[idx]
    f = finetuned_full.iloc[idx]

    print("\n" + "=" * 100)
    print(f"EXAMPLE {idx}")
    print("=" * 100)

    print("\nTOPIC:", b["topic"])
    print("DIFFICULTY:", b["difficulty"])

    print("\nQUESTION:")
    print(b["instruction"])

    print("\n--- BASE MODEL ---")
    print(b["prediction"])

    print("\n--- FINE-TUNED MODEL ---")
    print(f["prediction"])

    print("\n--- REFERENCE ---")
    print(b["reference"])

    print(
        f"\nROUGE-L: "
        f"{error_analysis.iloc[idx]['baseline_rougeL']:.4f}"
        f" → "
        f"{error_analysis.iloc[idx]['finetuned_rougeL']:.4f}"
    )


EXAMPLE 45

TOPIC: Dataset contamination detection (train/test overlap, benchmark leakage)
DIFFICULTY: senior

QUESTION:
Review this `data_split.py` for potential issues. I'm using `huggingface/datasets`.
```python
from datasets import load_dataset
dataset = load_dataset('my_corpus', split='train')
train_test_split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = train_test_split['train']
test_dataset = train_test_split['test']
# Later, I'm taking a subset of 'train_dataset' for validation.
val_dataset = train_dataset.train_test_split(test_size=0.1)['test']
```

--- BASE MODEL ---
Your code looks mostly correct and well-structured. However, there is one small issue that could be improved:

### Issue:
The line where you define the `val_dataset` should use the `'validation'` split instead of `'test'`. This is because when splitting data into training, validation, and test sets, Hugging Face's `datasets` library typically uses `'validation'` as the name for the validat

In [ ]:
# Save difficulty analysis
difficulty_path = os.path.join(
    results_dir,
    "difficulty_analysis.csv"
)

difficulty_analysis.to_csv(
    difficulty_path,
    index=False
)

# Save topic analysis
topic_path = os.path.join(
    results_dir,
    "topic_analysis.csv"
)

topic_analysis.to_csv(
    topic_path,
    index=False
)

# Save per-example error analysis
error_analysis_path = os.path.join(
    results_dir,
    "per_example_error_analysis.csv"
)

error_analysis.to_csv(
    error_analysis_path,
    index=False
)

print("✅ Analysis files saved")
print(difficulty_path)
print(topic_path)
print(error_analysis_path)

✅ Analysis files saved
/content/drive/MyDrive/LLM_FineTuning_Project/results/difficulty_analysis.csv
/content/drive/MyDrive/LLM_FineTuning_Project/results/topic_analysis.csv
/content/drive/MyDrive/LLM_FineTuning_Project/results/per_example_error_analysis.csv


## Final Results

The notebook above contains the complete fine-tuning pipeline, baseline vs. fine-tuned evaluation, per-example error analysis, and saved result files.

The final evaluation contains 161 test examples, with regression cases analyzed separately.